In [21]:
%pip install python-dotenv requests

Note: you may need to restart the kernel to use updated packages.


In [22]:
from getpass import getpass

key = getpass("Paste your AT API key: ")
with open(".env", "w") as f:
    f.write(f"AT_API_KEY={key}\n")
with open(".gitignore", "a") as f:
    f.write(".env\n")
print("Done")

Paste your AT API key:  ········


Done


In [23]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
k = os.getenv("AT_API_KEY")
print("Key loaded:", k is not None, "| length:", len(k) if k else 0)

Key loaded: True | length: 32


In [24]:
import os, json, requests
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("AT_API_KEY")

url = "https://api.at.govt.nz/realtime/legacy/"
r = requests.get(url, headers={"Ocp-Apim-Subscription-Key": key}, timeout=30)
print("Status:", r.status_code)

if r.status_code != 200:
    print(r.text[:500])
else:
    data = r.json()
    print("Top-level keys:", list(data.keys()))
    resp = data.get("response", data)
    entities = resp.get("entity", [])
    print("Number of entities:", len(entities))
    for e in entities:
        if "trip_update" in e or "TripUpdate" in e:
            print(json.dumps(e, indent=2)[:2500])
            break
    else:
        if entities:
            print("First entity keys:", list(entities[0].keys()))
            print(json.dumps(entities[0], indent=2)[:2500])

Status: 200
Top-level keys: ['status', 'response', 'error']
Number of entities: 3859
{
  "id": "257-870005-29160-2-A038B041100-30e84873",
  "trip_update": {
    "trip": {
      "trip_id": "257-870005-29160-2-A038B041100-30e84873",
      "start_time": "08:06:00",
      "start_date": "20261029",
      "schedule_relationship": 0,
      "route_id": "S-C-201",
      "direction_id": 0
    },
    "stop_time_update": {
      "stop_sequence": 22,
      "departure": {
        "delay": 36,
        "time": 1793219196,
        "uncertainty": 0
      },
      "stop_id": "9106-cca709e4",
      "schedule_relationship": 0
    },
    "vehicle": {
      "id": "59208",
      "label": "AMP        1208"
    },
    "timestamp": 1793219406,
    "delay": -11
  },
  "is_deleted": false
}


In [25]:
%%writefile collector.py
import os, time, sqlite3, logging
import requests
from dotenv import load_dotenv

load_dotenv(override=True)
KEY = os.getenv("AT_API_KEY")
URL = "https://api.at.govt.nz/realtime/legacy/"
DB = "bus_delays.db"
POLL_SECONDS = 60

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

SCHEMA = """
CREATE TABLE IF NOT EXISTS trip_delays (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    polled_at INTEGER,
    feed_ts INTEGER,
    trip_id TEXT,
    route_id TEXT,
    direction_id INTEGER,
    start_date TEXT,
    start_time TEXT,
    vehicle_id TEXT,
    stop_sequence INTEGER,
    stop_id TEXT,
    trip_delay INTEGER,
    stop_delay INTEGER,
    schedule_rel INTEGER,
    UNIQUE (trip_id, start_date, feed_ts)
);
CREATE INDEX IF NOT EXISTS idx_route_ts ON trip_delays (route_id, feed_ts);
"""

INSERT = """
INSERT OR IGNORE INTO trip_delays
(polled_at, feed_ts, trip_id, route_id, direction_id, start_date, start_time,
 vehicle_id, stop_sequence, stop_id, trip_delay, stop_delay, schedule_rel)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
"""

def parse(entities, polled_at):
    rows = []
    for e in entities:
        tu = e.get("trip_update")
        if not tu:
            continue
        trip = tu.get("trip") or {}
        stu = tu.get("stop_time_update") or {}
        if isinstance(stu, list):
            stu = stu[0] if stu else {}
        ev = stu.get("arrival") or stu.get("departure") or {}
        veh = tu.get("vehicle") or {}
        rows.append((
            polled_at, tu.get("timestamp"), trip.get("trip_id"),
            trip.get("route_id"), trip.get("direction_id"),
            trip.get("start_date"), trip.get("start_time"),
            veh.get("id"), stu.get("stop_sequence"), stu.get("stop_id"),
            tu.get("delay"), ev.get("delay"), trip.get("schedule_relationship"),
        ))
    return rows

def main():
    con = sqlite3.connect(DB)
    con.executescript(SCHEMA)
    logging.info("Collector started. Press Ctrl+C to stop.")
    while True:
        start = time.time()
        try:
            r = requests.get(URL, headers={"Ocp-Apim-Subscription-Key": KEY}, timeout=30)
            r.raise_for_status()
            entities = (r.json().get("response") or {}).get("entity") or []
            rows = parse(entities, int(start))
            con.executemany(INSERT, rows)
            con.commit()
            logging.info("Saved %d trip updates", len(rows))
        except Exception as ex:
            logging.warning("Poll failed: %s", ex)
        time.sleep(max(1, POLL_SECONDS - (time.time() - start)))

if __name__ == "__main__":
    main()

Overwriting collector.py


In [26]:
import os
print(os.path.exists("collector.py"))

True


In [27]:
print(os.getcwd())

C:\Users\Aum Shukla\akl-bus-delays


In [28]:
import sqlite3, pandas as pd
con = sqlite3.connect("bus_delays.db")
print(pd.read_sql("SELECT COUNT(*) AS rows FROM trip_delays", con))
print(pd.read_sql("SELECT * FROM trip_delays LIMIT 5", con))

     rows
0  165010
   id   polled_at     feed_ts                                  trip_id  \
0   1  1792655691  1792655389  257-870008-69960-2-B172A177100-e5884c4c   
1   2  1792655691  1792655525  257-870005-72360-2-A174B185110-9be74534   
2   3  1792655691  1792652095  257-870006-66360-2-B162A167110-1677f651   
3   4  1792655691  1792599127  257-870056-78060-2-B182A987110-dbaaaed5   
4   5  1792655691  1792655396  257-870005-74160-2-A176B187130-154cb43b   

  route_id  direction_id start_date start_time vehicle_id  stop_sequence  \
0  S-C-201             1   20261022   19:26:00      59428             22   
1  S-C-201             0   20261022   20:06:00      59157             12   
2  S-C-201             1   20261022   18:26:00      59157             23   
3  S-C-201             1   20261022   21:41:00      59157              1   
4  S-C-201             0   20261022   20:36:00      59155              3   

         stop_id  trip_delay  stop_delay  schedule_rel  
0  9237-f5d0ce7f     

In [29]:
import sqlite3, pandas as pd
con = sqlite3.connect("bus_delays.db")
print(pd.read_sql("""
    SELECT COUNT(DISTINCT polled_at) AS polls,
           COUNT(DISTINCT route_id) AS routes,
           COUNT(DISTINCT trip_id) AS trips,
           datetime(MIN(polled_at),'unixepoch','+13 hours') AS first_poll_nz,
           datetime(MAX(polled_at),'unixepoch','+13 hours') AS last_poll_nz
    FROM trip_delays
""", con))

   polls  routes  trips        first_poll_nz         last_poll_nz
0    391     518  10623  2026-10-22 20:54:51  2026-10-28 16:24:25


In [30]:
import sqlite3, pandas as pd
con = sqlite3.connect("bus_delays.db")
polls = pd.read_sql("SELECT DISTINCT polled_at FROM trip_delays ORDER BY polled_at", con)
polls["nz_time"] = pd.to_datetime(polls["polled_at"], unit="s", utc=True).dt.tz_convert("Pacific/Auckland")
print(polls.groupby(polls["nz_time"].dt.floor("h")).size())
print("Largest gaps (minutes):")
print((polls["polled_at"].diff().sort_values(ascending=False).head(5) / 60).round(0))

nz_time
2026-10-22 20:00:00+13:00     3
2026-10-23 08:00:00+13:00    43
2026-10-23 09:00:00+13:00    60
2026-10-23 10:00:00+13:00    60
2026-10-23 11:00:00+13:00    60
2026-10-23 12:00:00+13:00    60
2026-10-23 13:00:00+13:00    60
2026-10-23 14:00:00+13:00    24
2026-10-28 16:00:00+13:00    21
dtype: int64
Largest gaps (minutes):
370    7301.0
3       681.0
1         1.0
258       1.0
267       1.0
Name: polled_at, dtype: float64


In [31]:
import requests, zipfile, io, os

r = requests.get("https://gtfs.at.govt.nz/gtfs.zip", timeout=120)
print(r.status_code, round(len(r.content) / 1e6, 1), "MB")
os.makedirs("gtfs", exist_ok=True)
zipfile.ZipFile(io.BytesIO(r.content)).extractall("gtfs")
print(os.listdir("gtfs"))

routes = pd.read_csv("gtfs/routes.txt")
print(routes["route_type"].value_counts())   # 3 = bus, 2 = rail, 4 = ferry
cal = pd.read_csv("gtfs/calendar.txt")
print("Service dates:", cal["start_date"].min(), "to", cal["end_date"].max())

200 26.7 MB
['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'fare_attributes.txt', 'fare_rules.txt', 'feed_info.txt', 'frequencies.txt', 'routes.txt', 'shapes.txt', 'stops.txt', 'stop_times.txt', 'transfers.txt', 'trips.txt']
route_type
3    197
4     17
2      4
Name: count, dtype: int64
Service dates: 20261012 to 20270207


In [32]:
rt = pd.read_sql("SELECT DISTINCT route_id FROM trip_delays", con)
m = rt.merge(routes[["route_id", "route_short_name", "route_type"]], on="route_id", how="left")
print("Matched fraction:", round(m["route_short_name"].notna().mean(), 3))
print(m["route_type"].value_counts(dropna=False))

Matched fraction: 0.39
route_type
NaN    316
3.0    189
4.0     10
2.0      3
Name: count, dtype: int64


In [33]:
unmatched = m[m["route_short_name"].isna()]["route_id"]
print("Unmatched live route_ids:", unmatched.head(15).tolist())
print("Static route_ids:        ", routes["route_id"].head(15).tolist())

trips = pd.read_csv("gtfs/trips.txt", usecols=["trip_id", "route_id"])
rt_trips = pd.read_sql("SELECT DISTINCT trip_id FROM trip_delays", con)
print("Trip ID match fraction:", round(rt_trips["trip_id"].isin(trips["trip_id"]).mean(), 3))

cal = pd.read_csv("gtfs/calendar.txt")
print("Service dates:", cal["start_date"].min(), "to", cal["end_date"].max())

Unmatched live route_ids: ['S001-203', 'S001E-203', 'S001F-203', 'S001N-203', 'S002-218', 'S002A-203', 'S002D-203', 'S002F-203', 'S003-203', 'S003C-203', 'S003D-203', 'S004F-203', 'S005-218', 'S005B-203', 'S005C-203']
Static route_ids:         ['101-202', '186-203', '191-203', '105-202', '106-202', '195-203', '111-203', '112-203', '114-203', '392-203', '115-203', '116-203', '11T-207', '11W-207', '12-207']
Trip ID match fraction: 0.951
Service dates: 20261012 to 20270207


In [34]:
import numpy as np

trips = pd.read_csv("gtfs/trips.txt", usecols=["trip_id", "route_id", "trip_headsign"]).rename(columns={"route_id": "static_route_id"})
routes = pd.read_csv("gtfs/routes.txt", usecols=["route_id", "route_short_name", "route_type"]).rename(columns={"route_id": "static_route_id"})
tr = trips.merge(routes, on="static_route_id", how="left")

df = pd.read_sql("SELECT * FROM trip_delays", con)
print("Raw rows:", len(df))

df = df.merge(tr, on="trip_id", how="left")
print("Trip matched:", round(df["route_short_name"].notna().mean(), 3))

df = df[df["route_type"] == 3].copy()
print("Bus rows:", len(df), "| bus routes:", df["route_short_name"].nunique())

# How stale is each reading when we polled it?
df["age_s"] = df["polled_at"] - df["feed_ts"]
print(df["age_s"].describe(percentiles=[.5, .75, .9, .95]).round(0))
print("Delay (s):")
print(df["trip_delay"].describe(percentiles=[.01, .5, .99]).round(0))

Raw rows: 165010
Trip matched: 0.988
Bus rows: 156907 | bus routes: 189
count    156907.0
mean         58.0
std         274.0
min           0.0
50%          24.0
75%          39.0
90%          54.0
95%          61.0
max       11334.0
Name: age_s, dtype: float64
Delay (s):
count    156907.0
mean         23.0
std         250.0
min       -3245.0
1%         -696.0
50%          35.0
99%         657.0
max        9263.0
Name: trip_delay, dtype: float64


In [35]:
n0 = len(df)
clean = df[df["age_s"] <= 120]
print("After freshness filter:", len(clean), f"({len(clean)/n0:.1%})")
clean = clean[clean["schedule_rel"].fillna(0) == 0]          # drop cancelled trips
clean = clean[clean["trip_delay"].between(-900, 1800)].copy()
print("After delay clip:", len(clean), f"({len(clean)/n0:.1%})")
print("Exact-zero delays:", f"{(clean['trip_delay'] == 0).mean():.1%}")

clean["nz_time"] = pd.to_datetime(clean["feed_ts"], unit="s", utc=True).dt.tz_convert("Pacific/Auckland")
clean["hour"] = clean["nz_time"].dt.hour
clean["late5"] = (clean["trip_delay"] > 300).astype(int)    # more than 5 min late
clean["early1"] = (clean["trip_delay"] < -60).astype(int)   # more than 1 min early

by_hour = clean.groupby("hour").agg(
    readings=("trip_delay", "size"),
    median_delay_s=("trip_delay", "median"),
    pct_late5=("late5", "mean"),
    pct_early1=("early1", "mean"),
).round(3)
print(by_hour)

by_route = clean.groupby("route_short_name").agg(
    readings=("trip_delay", "size"),
    median_delay_s=("trip_delay", "median"),
    pct_late5=("late5", "mean"),
).query("readings >= 500").sort_values("pct_late5", ascending=False).round(3)
print(by_route.head(10))

After freshness filter: 153708 (98.0%)
After delay clip: 153173 (97.6%)
Exact-zero delays: 0.2%
      readings  median_delay_s  pct_late5  pct_early1
hour                                                 
8        21018             4.0      0.075       0.370
9        25001            17.0      0.061       0.326
10       21837            23.0      0.057       0.298
11       21786            42.0      0.094       0.265
12       21811            50.0      0.108       0.230
13       21804            48.0      0.100       0.247
14        8706            54.0      0.115       0.244
16       10409            58.0      0.177       0.277
20         801            42.0      0.086       0.232
                  readings  median_delay_s  pct_late5
route_short_name                                     
981                    518           197.0      0.388
712                    706           163.0      0.344
856                   1543           200.0      0.330
13                    1921           165

In [36]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

d = clean[["route_short_name", "direction_id", "hour", "stop_sequence", "trip_delay", "trip_id"]].dropna().copy()
d["route_code"] = d["route_short_name"].astype("category").cat.codes
feats = ["route_code", "direction_id", "hour", "stop_sequence"]

# Split by trip, so readings of the same trip never sit on both sides
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(d, groups=d["trip_id"]))
train, test = d.iloc[tr_idx], d.iloc[te_idx]

# Baseline 1: one global median for everything
global_med = train["trip_delay"].median()
b1 = np.full(len(test), global_med)

# Baseline 2: median delay per route
route_med = train.groupby("route_short_name")["trip_delay"].median()
b2 = test["route_short_name"].map(route_med).fillna(global_med)

# Model: gradient boosting with route as a categorical feature
model = HistGradientBoostingRegressor(
    categorical_features=[True, False, False, False],
    max_iter=200, learning_rate=0.08, random_state=42,
)
model.fit(train[feats], train["trip_delay"])
pred = model.predict(test[feats])

print("Train rows:", len(train), "| Test rows:", len(test))
print("MAE global median (s):", round(mean_absolute_error(test["trip_delay"], b1), 1))
print("MAE route median  (s):", round(mean_absolute_error(test["trip_delay"], b2), 1))
print("MAE gradient boost(s):", round(mean_absolute_error(test["trip_delay"], pred), 1))

Train rows: 115566 | Test rows: 37607
MAE global median (s): 168.7
MAE route median  (s): 160.6
MAE gradient boost(s): 137.1


In [37]:
from sklearn.inspection import permutation_importance

sample = test.sample(8000, random_state=42)
r = permutation_importance(model, sample[feats], sample["trip_delay"],
                           scoring="neg_mean_absolute_error", n_repeats=5, random_state=42)
for name, imp in sorted(zip(feats, r.importances_mean), key=lambda x: -x[1]):
    print(f"{name:15s} +{imp:6.1f} s worse when shuffled")

route_code      +  60.8 s worse when shuffled
stop_sequence   +  37.3 s worse when shuffled
direction_id    +  13.2 s worse when shuffled
hour            +  11.2 s worse when shuffled


In [38]:
import os, json, joblib

cats = list(d["route_short_name"].astype("category").cat.categories)
meta = {
    "route_names": cats,
    "median_stop_seq": d.groupby("route_short_name")["stop_sequence"].median().to_dict(),
    "hours_seen": sorted(int(h) for h in d["hour"].unique()),
    "trained_rows": int(len(d)),
    "mae_seconds": 125.4,
}
os.makedirs("api", exist_ok=True)
joblib.dump(model, "api/model.joblib")
json.dump(meta, open("api/meta.json", "w"))
print("Saved", len(cats), "routes; hours seen:", meta["hours_seen"])

Saved 188 routes; hours seen: [8, 9, 10, 11, 12, 13, 14, 16, 20]


In [39]:
from getpass import getpass

key = getpass("Paste your AT API key: ").strip()
with open(".env", "w") as f:
    f.write(f"AT_API_KEY={key}\n")
print("Saved. Length:", len(key))

Paste your AT API key:  ········


Saved. Length: 32
